# 03 · Event Extraction

**Purpose:** Label each (article, ticker) pair with structured event types using
rule-based pattern matching, plus negation and speculation detection.

**Event types:** `ma` · `earnings` · `management` · `legal`

**Input:** `articles_mapped.parquet`  
**Output:** `events.parquet`  
Schema: `article_url, ticker, date, event_ma, event_earnings, event_management, event_legal, is_negated, is_speculative`

In [1]:
import re
import sys
from pathlib import Path

import pandas as pd

NB_DIR = Path().resolve()
sys.path.insert(0, str(NB_DIR))
from utils import load, save

In [2]:
articles_mapped = load("articles_mapped")
print(f"{len(articles_mapped):,} (article, ticker) pairs")

FileNotFoundError: articles_mapped.parquet not found — run the earlier notebook first.

## 1 · Pattern dictionaries

In [ ]:
# Each list entry is a compiled regex pattern
EVENT_PATTERNS: dict[str, list[re.Pattern]] = {
    "ma": [
        re.compile(p, re.IGNORECASE) for p in [
            r"\bacquisition\b", r"\bfusion\b", r"\brachat\b",
            r"\bOPA\b", r"\bcession\b", r"\boffre publique\b",
            r"\bprise de participation\b", r"\bmerger\b",
            r"\btakeover\b", r"\bconsortium\b",
        ]
    ],
    "earnings": [
        re.compile(p, re.IGNORECASE) for p in [
            r"\brésultats?\b", r"\bbénéfice[s]?\b", r"\bEBIT\b",
            r"\bEBITDA\b", r"\bchiffre d.affaires\b", r"\brevenus?\b",
            r"\bdividende[s]?\b", r"\bbénéfice net\b", r"\bmarge\b",
            r"\bperte[s]?\b", r"\bprofit[s]?\b", r"\bearnings\b",
        ]
    ],
    "management": [
        re.compile(p, re.IGNORECASE) for p in [
            r"\bPDG\b", r"\bdirecteur.général\b", r"\bnomination\b",
            r"\bdémission\b", r"\bconseil d.administration\b",
            r"\bprésidence\b", r"\bnommé\b", r"\bCEO\b",
            r"\bretraite\b", r"\bcomité exécutif\b",
        ]
    ],
    "legal": [
        re.compile(p, re.IGNORECASE) for p in [
            r"\bprocès\b", r"\blitige\b", r"\bsanction\b",
            r"\bamende\b", r"\btribunal\b", r"\brégulateur\b",
            r"\bpoursuite[s]?\b", r"\bplainte\b", r"\benquête\b",
            r"\bfraud[e]?\b", r"\bcorruption\b",
        ]
    ],
}

NEGATION_PATTERNS = [
    re.compile(p, re.IGNORECASE) for p in [
        r"\bne\b.{0,25}\bpas\b", r"\baucun\b", r"\bpas de\b",
        r"\bn.a pas\b", r"\bn.ont pas\b", r"\bjamais\b", r"\bsans\b",
    ]
]

SPECULATION_PATTERNS = [
    re.compile(p, re.IGNORECASE) for p in [
        r"\bpourrait\b", r"\benvisage\b", r"\bprévoit\b",
        r"\bselon des sources\b", r"\brumeur\b", r"\bdevrait\b",
        r"\bserait\b", r"\bnégociation[s]?\b", r"\bdiscussion[s]?\b",
        r"\bà l.étude\b", r"\bselon nos informations\b",
    ]
]

print("Patterns loaded.")

Patterns loaded.


## 2 · Detection logic

In [ ]:
def _any_match(text: str, patterns: list[re.Pattern]) -> bool:
    return any(p.search(text) for p in patterns)


def extract_events(text: str) -> dict:
    row: dict = {}
    for event_type, patterns in EVENT_PATTERNS.items():
        row[f"event_{event_type}"] = _any_match(text, patterns)
    row["is_negated"]    = _any_match(text, NEGATION_PATTERNS)
    row["is_speculative"] = _any_match(text, SPECULATION_PATTERNS)
    return row

## 3 · Apply to dataset

In [ ]:
event_cols = articles_mapped["text"].apply(extract_events).apply(pd.Series)

events = pd.concat(
    [articles_mapped[["article_url", "ticker", "date", "published_at"]].reset_index(drop=True),
     event_cols.reset_index(drop=True)],
    axis=1,
)

bool_cols = ["event_ma", "event_earnings", "event_management", "event_legal",
             "is_negated", "is_speculative"]

print("Event coverage (% of (article, ticker) pairs):")
print((events[bool_cols].mean() * 100).round(1).to_string())

Event coverage (% of (article, ticker) pairs):
event_ma             1.8
event_earnings      18.5
event_management    10.0
event_legal          4.2
is_negated          26.0
is_speculative      15.1


In [ ]:

has_any = events[bool_cols[:4]].any(axis=1)
print(f"Rows with ≥1 event:  {has_any.sum():,} / {len(events):,} ({has_any.mean()*100:.1f}%)")
# Co-occurrence matrix
events[bool_cols[:4]].astype(int).corr().round(2)

Rows with ≥1 event:  1,122 / 3,557 (31.5%)


,event_ma,event_earnings,event_management,event_legal
event_ma,1.00,0.00,0.05,-0.03
event_earnings,0.00,1.00,-0.06,0.01
event_management,0.05,-0.06,1.00,-0.05
event_legal,-0.03,0.01,-0.05,1.00


## 4 · Merge LLM event_type

The RSS pipeline's LLM analyzer assigns a structured `event_type` to every article.
Here we merge those LLM labels onto the events table so both the rule-based and
LLM-based signals coexist as features.

In [ ]:
sentiments = load("sentiments")

# Keep only articles that have an LLM event_type assigned
llm_event_types = sentiments[sentiments["event_type"].notna()][["article_url", "event_type", "event_fingerprint"]]
print(f"Articles with LLM event_type: {len(llm_event_types):,} / {len(sentiments):,}")
print("\nLLM event_type distribution:")
print(llm_event_types["event_type"].value_counts().to_string())

# Merge into events table (left join — rule-based rows keep their boolean flags)
events = events.merge(llm_event_types, on="article_url", how="left")
events["event_type"] = events["event_type"].fillna("other")
print(f"\nEvents table shape after merge: {events.shape}")

## 5 · Save updated events

In [ ]:
save(events, "events")

## 6 · Event signal visuals

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})

RULE_COLS = ["event_ma", "event_earnings", "event_management", "event_legal"]

# ── Panel A: LLM event_type distribution ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Event Signal Overview", fontsize=14, fontweight="bold")

llm_dist = events["event_type"].value_counts()
llm_dist.sort_values().plot.barh(ax=axes[0], color="coral")
axes[0].set_title("LLM event_type (all (article,ticker) pairs)")
axes[0].set_xlabel("Count")

# ── Panel B: Rule-based vs LLM coverage ───────────────────────────────────────
rule_coverage  = events[RULE_COLS].any(axis=1).mean() * 100
llm_coverage   = (events["event_type"] != "other").mean() * 100

axes[1].bar(
    ["Rule-based\n(any event)", "LLM\n(non-other)"],
    [rule_coverage, llm_coverage],
    color=["steelblue", "coral"],
)
axes[1].set_title("Coverage: % of rows with an event signal")
axes[1].set_ylabel("% of (article, ticker) pairs")
axes[1].set_ylim(0, 100)
for i, v in enumerate([rule_coverage, llm_coverage]):
    axes[1].text(i, v + 1, f"{v:.1f}%", ha="center", fontweight="bold")

# ── Panel C: Rule-based event type breakdown ───────────────────────────────────
rule_totals = events[RULE_COLS].sum().rename(
    index={"event_ma": "M&A", "event_earnings": "Earnings",
           "event_management": "Management", "event_legal": "Legal"}
)
rule_totals.sort_values().plot.barh(ax=axes[2], color="steelblue")
axes[2].set_title("Rule-based event counts")
axes[2].set_xlabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
# ── LLM event_type per company (top 15 tickers by LLM mentions) ───────────────
llm_events = events[events["event_type"] != "other"].copy()

if len(llm_events) > 0:
    top_tickers = llm_events["ticker"].value_counts().head(15).index
    pivot = (
        llm_events[llm_events["ticker"].isin(top_tickers)]
        .groupby(["ticker", "event_type"])
        .size()
        .unstack(fill_value=0)
    )
    # Normalise by row for a clearer heatmap
    pivot_norm = pivot.div(pivot.sum(axis=1), axis=0)

    fig, ax = plt.subplots(figsize=(14, 6))
    im = ax.imshow(pivot_norm.values, aspect="auto", cmap="Blues")
    ax.set_xticks(range(len(pivot_norm.columns)))
    ax.set_xticklabels(pivot_norm.columns, rotation=40, ha="right", fontsize=9)
    ax.set_yticks(range(len(pivot_norm.index)))
    ax.set_yticklabels(pivot_norm.index, fontsize=9)
    ax.set_title("LLM event_type share per company (top 15 tickers by LLM event count)")
    plt.colorbar(im, ax=ax, label="Share of LLM events")
    plt.tight_layout()
    plt.show()
else:
    print("No non-other LLM events found — ensure sentiments table has event_type populated.")